## Data Processing – Replicating “I Will Survive: Predicting Business Failures from Customer Ratings”

The following pipeline mirrors the data preparation and analysis steps of the Marketing Science case study and stores them in a Pickle file for further analysis.


In [1]:
import sys
from pathlib import Path

# add projects root directory to the system path to enable importing custom modules (e.g., from the "helpers" folder).
sys.path.append(str(Path("..").resolve()))

# Imports
import pandas as pd
import numpy as np


SEED = 555  # random seed for reproducability (change seed, if desired)
SET_ORIGINAL_INDICES = False  # if set to true, the original paper indices are selected

np.random.seed(SEED)

In [2]:
from constants import DATA_FOLDER

# Dataframes
reviews = pd.read_csv(DATA_FOLDER / "reviews.csv")
business_covariates = pd.read_csv(DATA_FOLDER / "business_covariates.csv")

In [3]:
# Datatypes of both datasets provided


display(business_covariates.dtypes)
display(reviews.dtypes)

display(business_covariates.head(2))
display(reviews.head(2))

# Unique user count
len(reviews["user_id"].unique())

business_id                 object
name                        object
neighborhood               float64
address                     object
city                        object
state                       object
postal_code                float64
latitude                   float64
longitude                  float64
stars                      float64
review_count                 int64
is_open                      int64
categories                  object
Checkin                      int64
chain                        int64
density                      int64
TRAIN                        int64
category                    object
FT                            bool
Price.Level                float64
Restaurant.Size            float64
Number.of.Seats            float64
ZRI                        float64
Distance.To.City.Centre    float64
dtype: object

review_id          object
business_id        object
user_id            object
date               object
stars               int64
text               object
funny               int64
cool                int64
useful              int64
Year                int64
first_year          int64
.groups            object
n_reviews           int64
AggRat            float64
language           object
tempcontiguity      int64
sentimenttext     float64
wordCount           int64
dtype: object

,business_id,name,neighborhood,address,city,state,postal_code,latitude,longitude,stars,...,chain,density,TRAIN,category,FT,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre
0,QkG3KUXwqZBW18A9k1xqCA,"""Red Lobster""",NaN,"""2810 North 75th Ave""",Phoenix,AZ,85035.0,33.478735,-112.221379,2.5,...,1,11,0,American,False,2.0,700.0,350.0,1248.0,14035.314000
1,5XejqzaFmtkZMstJS5Iy-w,"""D'Lish Cafe""",NaN,"""503 W Thomas Rd""",Phoenix,AZ,85013.0,33.480301,-112.080586,4.0,...,0,17,0,American,False,1.0,200.0,30.0,1557.0,3279.013976


,review_id,business_id,user_id,date,stars,text,funny,cool,useful,Year,first_year,.groups,n_reviews,AggRat,language,tempcontiguity,sentimenttext,wordCount
0,EQF-SyHb_Yg0HC9E9BppOg,--g-a85VwrdZJNf0R95GcQ,SvsoiaCf0WG7UIDJOxJ7Yg,2013-11-14,5,super fresh food..great prices. ala carte and ...,0,1,3,2013,2013,drop,24,5.0,en,0,0.461132,10
1,qfvzHEL0gGxRdTo7NB2fnw,--g-a85VwrdZJNf0R95GcQ,D0YggUjK98hwIpiqXA9y_g,2013-11-18,5,I was in the Kabab House for the first time on...,0,1,3,2013,2013,drop,24,5.0,en,0,0.287281,44


41301

In [4]:
# Get range of reviews

min_date = pd.to_datetime(reviews["date"]).min()
second_oldest_date = pd.to_datetime(reviews["date"]).sort_values().iloc[1]
max_date = pd.to_datetime(reviews["date"]).max()
print(f"Minimum review date: {min_date}")
print(f"Maximum review date: {max_date}")

Minimum review date: 2010-01-03 00:00:00
Maximum review date: 2017-12-11 00:00:00


In [5]:
# create indices for training evaluation and calibration

from constants import CALIBRATION_INDICES, EVAL_INDICES, TRAIN_INDICES

# get original indices
if SET_ORIGINAL_INDICES:
    train_indices = TRAIN_INDICES
    calibration_indices = CALIBRATION_INDICES
    eval_indices = EVAL_INDICES

# get indices based on random seed
else:
    indices = np.random.permutation(len(business_covariates))
    indices_val_cal = np.random.permutation(
        np.arange(500, len(business_covariates))
    )  # range 500-921 (because of sorting)

    train_indices = indices[:500]  # take 500 random samples
    calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
    eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

In [6]:
assert (business_covariates.get("TRAIN")).sum() == 0, "training set already assigned!"

# set 'TRAIN' variable to 1 for train_indices, 0 otherwise
business_covariates.loc[train_indices, "TRAIN"] = 1

# sort business_covariates so that rows with Train==1 come first
business_covariates = business_covariates.sort_values(
    by="TRAIN", ascending=False
).reset_index(
    drop=True
)  # it is possible to retreive all training data with :500

n_train = len(train_indices)  # number of training samples (500)

print(f"Train/Calibration/Eval indices created:")
print(f"  Train: {len(train_indices)} samples")
print(f"  Calibration: {len(calibration_indices)} samples")
print(f"  Eval: {len(eval_indices)} samples")

Train/Calibration/Eval indices created:
  Train: 500 samples
  Calibration: 100 samples
  Eval: 321 samples


In [7]:
# initialize list with data needed for stan

ratings = []
sentiment = []
days = []
time = []
age = []

business_ids = business_covariates["business_id"].values

# convert date string into datetime object
reviews["date"] = pd.to_datetime(reviews["date"])

for k, business_id in enumerate(business_ids):
    if k % 100 == 0:
        print(f"[{k}] - Conversion for business_id: {business_id}")

    # get temporary dataframe of all reviews with given business_id and assign column 'Number' (Rating 0, ..., M_i)
    df_temp = (
        reviews[reviews["business_id"] == business_id]
        .reset_index(drop=True)
        .assign(Number=lambda x: x.index)
    )

    # calculate days since first review
    df_temp["Days"] = (df_temp["date"] - df_temp["date"].iloc[0]).dt.days
    days.extend(df_temp["Days"].tolist())
    sentiment.extend(df_temp["sentimenttext"].tolist())
    ratings.extend(df_temp["stars"].tolist())
    time.append(len(df_temp))
    age.append(df_temp["Days"].iloc[-1])


# validation checks
assert sum(time) == len(sentiment)
assert len(days) == len(sentiment)
assert len(ratings) == len(sentiment)
print("Done ... validation checks passed!")
print("Created required lists for MCMC sampling")

[0] - Conversion for business_id: 0CZ4opKkec9yAS8qmXoWdA
[100] - Conversion for business_id: XdhoZK1LYGYmmnUwpFNV-w
[200] - Conversion for business_id: k-NEUIXaPnXtrzCo1MeO7g
[300] - Conversion for business_id: MKvRBwtfIcH9206rzBeVxg
[400] - Conversion for business_id: ibOX3CypYVz0nJhCN5Wmcw
[500] - Conversion for business_id: PvufhE6-sy2pCTQu9yvjiw
[600] - Conversion for business_id: NUh5L9ZX-YrM5uAKhKQXoQ
[700] - Conversion for business_id: LVkILnI-bXtlCJggb3hfkg
[800] - Conversion for business_id: 6B-sZzuNYnVz5sk2c4EfHA
[900] - Conversion for business_id: v0TT3dcZ8OdTtaJEKxIcuA
Done ... validation checks passed!
Created required lists for MCMC sampling


In [8]:
assert (
    not "Age" in business_covariates
), "The key Age is already added to dataframe! Make sure, that you only run this cell once!"

# add restaurant age in days to dataframe
business_covariates["Age"] = age

# change checkin count to checkin rates (number of checkins every month, assuming a month contains 28 days)
business_covariates["Checkin"] = (
    business_covariates["Checkin"] / business_covariates["Age"] * 28
)

# add log of age to dataframe for later analysis
business_covariates["logAge"] = np.log(business_covariates["Age"])

# only get relevant covariates for training
relevant_covariates = business_covariates[
    [
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Age",
    ]
].copy()

relevant_covariates

,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,5,3.543424,American,0,2.0,90.0,22.0,1360.0,1209
1,22,0.253213,Cafes,0,2.0,63.0,0.0,1710.0,2101
2,4,5.920371,American,0,2.0,136.0,80.0,1515.0,2587
3,5,5.350254,American,0,2.0,840.0,500.0,1515.0,2758
4,4,0.224414,Cafes,0,2.0,91.0,0.0,1390.0,1622
...,...,...,...,...,...,...,...,...,...
916,63,10.618462,Asian,0,2.0,114.0,62.0,1495.0,1300
917,56,22.201183,American,0,2.0,289.0,160.0,1495.0,1014
918,2,2.121842,Mexican,0,1.0,147.0,68.0,1360.0,673
919,20,2.145985,Fast Food,1,1.0,251.0,150.0,1469.0,2192


In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

assert (
    len(relevant_covariates.columns) == 9
), "One Hot Coding already performed on dataframe!"

# Mark as categorial variable
relevant_covariates["category"] = relevant_covariates["category"].astype("category")

# One Hot Encoding
# also possible with OneHotEncoder and dropping first column (category_American), but wanting to retain the removed category (category_Other) in the original study
relevant_covariates_encoded = pd.get_dummies(
    relevant_covariates, columns=["category"], drop_first=False, dtype=int
)

###### Create covariate matrix with relevant covariates

# First two numeric columns before category column
first_numeric = ["density", "Checkin"]

# Category dummies (alphabetically sorted)
category_cols = sorted(
    [col for col in relevant_covariates_encoded.columns if col.startswith("category_")]
)

# Remaining numeric columns after category in original order
remaining_numeric = [
    "chain",
    "Price.Level",
    "Restaurant.Size",
    "Number.of.Seats",
    "ZRI",
    "Age",
]

# combine original order
column_order = first_numeric + category_cols + remaining_numeric
relevant_covariates = relevant_covariates_encoded[column_order]

# remove category_Other (8th column)
if len(relevant_covariates.columns) > 7:
    col_to_remove = relevant_covariates.columns[7]
    print(f"Removing column at index 7 (category_Other): '{col_to_remove}'")

    # Verify it's categoryOther
    if "Other" in col_to_remove:
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])
    else:
        # if not expected
        print(f"WARNING: Expected 'categoryOther' but found '{col_to_remove}'")
        print(f"All columns: {relevant_covariates.columns.tolist()}")

        # Still remove it
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])

print(f"Final covariate matrix shape: {relevant_covariates.shape}")
print(f"Columns: {relevant_covariates.columns.tolist()}")

relevant_covariates.head(1)

Removing column at index 7 (category_Other): 'category_Other'
Final covariate matrix shape: (921, 16)
Columns: ['density', 'Checkin', 'category_American', 'category_Asian', 'category_Cafes', 'category_Fast Food', 'category_Mexican', 'category_Pizza', 'category_Salad', 'category_Speciality Food', 'chain', 'Price.Level', 'Restaurant.Size', 'Number.of.Seats', 'ZRI', 'Age']


,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,5,3.543424,1,0,0,0,0,0,0,0,0,2.0,90.0,22.0,1360.0,1209


In [10]:
imputer = SimpleImputer(strategy="median")  # impute column with median value
scaler = StandardScaler(with_std=False)  # only centering, no scaling!

X_train = relevant_covariates.iloc[:n_train].copy()

# First fit imputer on training data (to get medians for each column)
imputer.fit(X_train)

# Then fit scaler on imputed training data (to get means for centering)
X_train_imputed = imputer.transform(X_train)
scaler.fit(X_train_imputed)

# Now apply both transformations to all data
cov_mat_imputed = imputer.transform(relevant_covariates)
cov_mat_preprocessed = scaler.transform(cov_mat_imputed)

X_train_preprocessed = cov_mat_preprocessed[:n_train]

# QR-decomposition
Q, R = np.linalg.qr(X_train_preprocessed)

# scale the Q and R matrix appropriately
Q_scaled = Q * np.sqrt(n_train - 1)
R_scaled = R / np.sqrt(n_train - 1)

X_test = cov_mat_preprocessed[n_train:]

display(X_train)
display(pd.DataFrame(cov_mat_preprocessed))

,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,5,3.543424,1,0,0,0,0,0,0,0,0,2.0,90.0,22.0,1360.0,1209
1,22,0.253213,0,0,1,0,0,0,0,0,0,2.0,63.0,0.0,1710.0,2101
2,4,5.920371,1,0,0,0,0,0,0,0,0,2.0,136.0,80.0,1515.0,2587
3,5,5.350254,1,0,0,0,0,0,0,0,0,2.0,840.0,500.0,1515.0,2758
4,4,0.224414,0,0,1,0,0,0,0,0,0,2.0,91.0,0.0,1390.0,1622
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,20,1.306329,0,0,1,0,0,0,0,0,0,1.0,120.0,42.0,1710.0,2765
496,13,11.603604,1,0,0,0,0,0,0,0,0,2.0,123.0,52.0,1557.0,555
497,8,9.153846,0,0,0,0,1,0,0,0,0,1.0,335.0,126.0,2007.0,624
498,3,1.523592,0,0,0,0,1,0,0,0,1,1.0,460.0,150.0,1287.0,2628


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-7.616,-1.608112,0.752,-0.136,-0.082,-0.116,-0.176,-0.092,-0.038,-0.042,-0.286,0.538,-131.092,-67.486,-130.56,-166.896
1,9.384,-4.898324,-0.248,-0.136,0.918,-0.116,-0.176,-0.092,-0.038,-0.042,-0.286,0.538,-158.092,-89.486,219.44,725.104
2,-8.616,0.768835,0.752,-0.136,-0.082,-0.116,-0.176,-0.092,-0.038,-0.042,-0.286,0.538,-85.092,-9.486,24.44,1211.104
3,-7.616,0.198717,0.752,-0.136,-0.082,-0.116,-0.176,-0.092,-0.038,-0.042,-0.286,0.538,618.908,410.514,24.44,1382.104
4,-8.616,-4.927122,-0.248,-0.136,0.918,-0.116,-0.176,-0.092,-0.038,-0.042,-0.286,0.538,-130.092,-89.486,-100.56,246.104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,50.384,5.466925,-0.248,0.864,-0.082,-0.116,-0.176,-0.092,-0.038,-0.042,-0.286,0.538,-107.092,-27.486,4.44,-75.896
917,43.384,17.049647,0.752,-0.136,-0.082,-0.116,-0.176,-0.092,-0.038,-0.042,-0.286,0.538,67.908,70.514,4.44,-361.896
918,-10.616,-3.029694,-0.248,-0.136,-0.082,-0.116,0.824,-0.092,-0.038,-0.042,-0.286,-0.462,-74.092,-21.486,-130.56,-702.896
919,7.384,-3.005551,-0.248,-0.136,-0.082,0.884,-0.176,-0.092,-0.038,-0.042,0.714,-0.462,29.908,60.514,-21.56,816.104


In [11]:
from helpers import comp_entropy

# aggregate review stats
review_stats = (
    reviews.groupby("business_id")
    .agg(
        VAR=("stars", "var"),
        MEAN=("stars", "mean"),
        ENTR=("stars", lambda x: comp_entropy(x)),
        COUNT=("stars", "size"),
        ONE_STAR=("stars", lambda x: (x == 1).sum()),
        TWO_STAR=("stars", lambda x: (x == 2).sum()),
        THREE_STAR=("stars", lambda x: (x == 3).sum()),
        FOUR_STAR=("stars", lambda x: (x == 4).sum()),
        FIVE_STAR=("stars", lambda x: (x == 5).sum()),
    )
    .reset_index()
)

# mutate count into probabilities
for col in ["ONE_STAR", "TWO_STAR", "THREE_STAR", "FOUR_STAR", "FIVE_STAR"]:
    review_stats[col] = review_stats[col] / review_stats["COUNT"]

# Add variation coeffient to review_stats
review_stats["COV"] = np.sqrt(review_stats["VAR"]) / review_stats["MEAN"]

# select covariates, that are relevant for training the benchmark models
benchmark_covariates = business_covariates[
    [
        "business_id",
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Distance.To.City.Centre",
        "Age",
        "is_open",
    ]
].copy()

# Add Closed column (opposite from is_open)
benchmark_covariates["Closed"] = 1 - benchmark_covariates["is_open"]

# merge covariates with aggregate review_stats
benchmark_covariates = benchmark_covariates.merge(
    review_stats, on="business_id", how="left"
)

# add logarithmic count
benchmark_covariates["l_COUNT"] = np.log(benchmark_covariates["COUNT"])

# Convert to categorical again
benchmark_covariates["category"] = benchmark_covariates["category"].astype("category")

# Convert Closed to categorical with proper labels (Closed = 1, Open = 0)
benchmark_covariates["Closed"] = (
    benchmark_covariates["Closed"].map({1: "Closed", 0: "Open"}).astype("category")
)

print(f"Benchmark covariates prepared with shape: {benchmark_covariates.shape}")

benchmark_covariates

/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Benchmark covariates prepared with shape: (921, 24)


,business_id,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre,...,MEAN,ENTR,COUNT,ONE_STAR,TWO_STAR,THREE_STAR,FOUR_STAR,FIVE_STAR,COV,l_COUNT
0,0CZ4opKkec9yAS8qmXoWdA,5,3.543424,American,0,2.0,90.0,22.0,1360.0,27645.693977,...,3.096774,1.554448,62,0.209677,0.225806,0.129032,0.129032,0.306452,0.505393,4.127134
1,2oQuTgkhY7QlVdfMa8cHgA,22,0.253213,Cafes,0,2.0,63.0,0.0,1710.0,7765.631452,...,4.800000,0.258641,35,0.028571,0.028571,0.000000,0.000000,0.942857,0.173570,3.555348
2,pu7maVMRHbIUv2x3B_xMHQ,4,5.920371,American,0,2.0,136.0,80.0,1515.0,11028.049606,...,2.978261,1.588121,138,0.246377,0.173913,0.137681,0.239130,0.202899,0.500934,4.927254
3,ohDvHtWVpCJ_6b1UrjGRPg,5,5.350254,American,0,2.0,840.0,500.0,1515.0,13927.105743,...,4.411765,1.007766,85,0.023529,0.047059,0.035294,0.282353,0.611765,0.213591,4.442651
4,WZVnmFXoE42coc4FmcbEDQ,4,0.224414,Cafes,0,2.0,91.0,0.0,1390.0,20605.187981,...,4.857143,0.302784,42,0.023810,0.000000,0.000000,0.047619,0.928571,0.133128,3.737670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,SLOk1JpV0JQK-MXYroDKYQ,63,10.618462,Asian,0,2.0,114.0,62.0,1495.0,203.254670,...,3.232323,1.592469,99,0.151515,0.181818,0.181818,0.252525,0.232323,0.430384,4.595120
917,roCxdzue-nME2akdIRPTfA,56,22.201183,American,0,2.0,289.0,160.0,1495.0,477.950899,...,3.822917,1.348241,96,0.052083,0.083333,0.135417,0.447917,0.281250,0.286534,4.564348
918,ZxO1vuGoiCzir1r9Ma3BfQ,2,2.121842,Mexican,0,1.0,147.0,68.0,1360.0,25823.621932,...,4.100000,1.192195,20,0.050000,0.000000,0.200000,0.300000,0.450000,0.261257,2.995732
919,gkIyPrNGpF4EGqoYWe2KBQ,20,2.145985,Fast Food,1,1.0,251.0,150.0,1469.0,18811.958072,...,2.500000,1.408351,26,0.461538,0.076923,0.153846,0.115385,0.192308,0.652380,3.258097


In [12]:
from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER

# Create ModelData instance with all data in one place
model_data = ModelData(
    n_states=None,  # not used yet, reserved for HMM models (prepare_stan_data function)
    n_total=len(time),
    n_train=n_train,
    n_obs=int(np.sum(time)),
    n_covs=cov_mat_preprocessed.shape[1],
    time=time,
    closed=1 - business_covariates["is_open"].values,
    days=days,
    ratings=ratings,
    sentiment=sentiment,
    Q=Q_scaled,
    R=R_scaled,
    X_test=X_test,
    imputer=imputer,
    scaler=scaler,
    train_indices=train_indices,
    calibration_indices=calibration_indices,
    eval_indices=eval_indices,
    business_covariates=business_covariates,
    cov_mat=cov_mat_preprocessed,
    benchmark_covariates=benchmark_covariates,
)

# save to pickle
if SET_ORIGINAL_INDICES:
    output_path = PROCESSED_DATA_FOLDER / f"processed_data_original.pkl"
else:
    output_path = PROCESSED_DATA_FOLDER / f"processed_data_{SEED}.pkl"

model_data.to_pickle(output_path)

print(f"Data saved to {output_path}")
print(model_data.summary())

Data saved to /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/processed/processed_data_555.pkl

ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 100
- Eval indices: 321

Benchmark data: Available
- Benchmark covariates shape: (921, 24)
- Columns: business_id, density, Checkin, category, chain...

